<a href="https://colab.research.google.com/github/lotus-outlook-6/Building-an-Auto-Encoder/blob/main/sae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoEncoders

## Importing the libraries

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

## Importing the dataset

In [5]:
movies = pd.read_csv('/content/drive/MyDrive/Udemy/Deep Learning A-Z 2026: Neural Networks, AI & ChatGPT Prize/Datasets/Deep Learning A-Z/Part 5 - Boltzmann Machines (BM)/ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
users = pd.read_csv('/content/drive/MyDrive/Udemy/Deep Learning A-Z 2026: Neural Networks, AI & ChatGPT Prize/Datasets/Deep Learning A-Z/Part 5 - Boltzmann Machines (BM)/ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
ratings = pd.read_csv('/content/drive/MyDrive/Udemy/Deep Learning A-Z 2026: Neural Networks, AI & ChatGPT Prize/Datasets/Deep Learning A-Z/Part 5 - Boltzmann Machines (BM)/ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')

## Preparing the training set and the test set

In [6]:
training_set = pd.read_csv('/content/drive/MyDrive/Udemy/Deep Learning A-Z 2026: Neural Networks, AI & ChatGPT Prize/Datasets/Deep Learning A-Z/Part 5 - Boltzmann Machines (BM)/ml-100k/u1.base', delimiter = '\t')
training_set = np.array(training_set, dtype = 'int')
test_set = pd.read_csv('/content/drive/MyDrive/Udemy/Deep Learning A-Z 2026: Neural Networks, AI & ChatGPT Prize/Datasets/Deep Learning A-Z/Part 5 - Boltzmann Machines (BM)/ml-100k/u1.test', delimiter = '\t')
test_set = np.array(test_set, dtype = 'int')

## Getting the number of users and movies

In [7]:
nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))

## Converting the data into an array with users in lines and movies in columns

In [8]:
def convert(data):
    new_data = []
    for id_users in range(1, nb_users + 1):
        id_movies = data[:,1][data[:,0] == id_users]
        id_ratings = data[:,2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings
        new_data.append(list(ratings))
    return new_data
training_set = convert(training_set)
test_set = convert(test_set)

## Converting the data into Torch tensors

In [9]:
training_set = torch.FloatTensor(training_set)
test_set = torch.FloatTensor(test_set)

## Creating the architecture of the Neural Network

In [10]:
class SAE(nn.Module):
    def __init__(self, ):
        super(SAE, self).__init__()
        self.fc1 = nn.Linear(nb_movies, 20)
        self.fc2 = nn.Linear(20, 10)
        self.fc3 = nn.Linear(10, 20)
        self.fc4 = nn.Linear(20, nb_movies)
        self.activation = nn.Sigmoid()
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x
sae = SAE()
criterion = nn.MSELoss()
optimizer = optim.RMSprop(sae.parameters(), lr = 0.01, weight_decay = 0.5)

## Training the SAE

In [11]:
nb_epoch = 200
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.
    for id_user in range(nb_users):
        input = Variable(training_set[id_user]).unsqueeze(0)
        target = input.clone()
        if torch.sum(target.data > 0) > 0:
            output = sae(input)
            target.require_grad = False
            output[target == 0] = 0
            loss = criterion(output, target)
            mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10)
            loss.backward()
            train_loss += np.sqrt(loss.data.item()*mean_corrector) # Corrected line
            s += 1.
            optimizer.step()
    print('epoch: '+str(epoch)+' loss: '+str(train_loss/s))

epoch: 1 loss: 1.771173716399781
epoch: 2 loss: 1.0967630558577093
epoch: 3 loss: 1.0534468010179285
epoch: 4 loss: 1.0383698704724706
epoch: 5 loss: 1.0308120708147577
epoch: 6 loss: 1.026320057859474
epoch: 7 loss: 1.0240784984523326
epoch: 8 loss: 1.0218265704129317
epoch: 9 loss: 1.0209867718461143
epoch: 10 loss: 1.0195420201359353
epoch: 11 loss: 1.0189092183585988
epoch: 12 loss: 1.0183495155298965
epoch: 13 loss: 1.0179692670406042
epoch: 14 loss: 1.017517277763342
epoch: 15 loss: 1.0172156781890969
epoch: 16 loss: 1.0166490698835153
epoch: 17 loss: 1.0168365236739376
epoch: 18 loss: 1.0165034896469265
epoch: 19 loss: 1.0163183657221344
epoch: 20 loss: 1.0160177783864754
epoch: 21 loss: 1.0163636978662618
epoch: 22 loss: 1.0160905503594306
epoch: 23 loss: 1.0158431939508308
epoch: 24 loss: 1.015923461875612
epoch: 25 loss: 1.0157639405887509
epoch: 26 loss: 1.0154266366003548
epoch: 27 loss: 1.0152603967620772
epoch: 28 loss: 1.014785829038176
epoch: 29 loss: 1.0138448208340147

In [12]:
nb_epoch = 200
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.
    for id_user in range(nb_users):
        input = Variable(training_set[id_user]).unsqueeze(0)
        target = input.clone()
        if torch.sum(target.data > 0) > 0:
            output = sae(input)
            target.require_grad = False
            output[target == 0] = 0
            loss = criterion(output, target)
            mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10)
            loss.backward()
            train_loss += np.sqrt(loss.data*mean_corrector)
            s += 1.
            optimizer.step()
    print('epoch: '+str(epoch)+' loss: '+str(train_loss/s))

/tmp/ipykernel_518/1130214824.py:15: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  train_loss += np.sqrt(loss.data*mean_corrector)


epoch: 1 loss: tensor(0.9139)
epoch: 2 loss: tensor(0.9139)
epoch: 3 loss: tensor(0.9139)
epoch: 4 loss: tensor(0.9134)
epoch: 5 loss: tensor(0.9131)
epoch: 6 loss: tensor(0.9128)
epoch: 7 loss: tensor(0.9145)
epoch: 8 loss: tensor(0.9126)
epoch: 9 loss: tensor(0.9124)
epoch: 10 loss: tensor(0.9121)
epoch: 11 loss: tensor(0.9121)
epoch: 12 loss: tensor(0.9116)
epoch: 13 loss: tensor(0.9116)
epoch: 14 loss: tensor(0.9120)
epoch: 15 loss: tensor(0.9118)
epoch: 16 loss: tensor(0.9116)
epoch: 17 loss: tensor(0.9111)
epoch: 18 loss: tensor(0.9108)
epoch: 19 loss: tensor(0.9105)
epoch: 20 loss: tensor(0.9101)
epoch: 21 loss: tensor(0.9101)
epoch: 22 loss: tensor(0.9098)
epoch: 23 loss: tensor(0.9093)
epoch: 24 loss: tensor(0.9089)
epoch: 25 loss: tensor(0.9082)
epoch: 26 loss: tensor(0.9082)
epoch: 27 loss: tensor(0.9084)
epoch: 28 loss: tensor(0.9078)
epoch: 29 loss: tensor(0.9074)
epoch: 30 loss: tensor(0.9072)
epoch: 31 loss: tensor(0.9074)
epoch: 32 loss: tensor(0.9069)
epoch: 33 loss: t

## Testing the SAE

In [13]:
test_loss = 0
s = 0.
for id_user in range(nb_users):
    input = Variable(training_set[id_user]).unsqueeze(0)
    target = Variable(test_set[id_user]).unsqueeze(0)
    if torch.sum(target.data > 0) > 0:
        output = sae(input)
        target.require_grad = False
        output[target == 0] = 0
        loss = criterion(output, target)
        mean_corrector = nb_movies/float(torch.sum(target.data > 0) + 1e-10)
        test_loss += np.sqrt(loss.data*mean_corrector)
        s += 1.
print('test loss: '+str(test_loss/s))

test loss: tensor(0.9453)


/tmp/ipykernel_518/1678710381.py:12: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  test_loss += np.sqrt(loss.data*mean_corrector)
